In [0]:
%sql

INSERT OVERWRITE proyecto_final.silver.hospitales (
  nombre_hospital,
  nombre_funcional_completo,
  nombre_generico,
  especialidad,
  tipo_atencion,
  direccion,
  barrio,
  fuente_datos,
  comuna,
  telefono,
  sitio_web,
  coordenadas,
  tipo_entidad,
  longitud,
  latitud
)
WITH datos_limpios AS (
  SELECT 
    LOWER(REPLACE(TRIM(nombre_hospital), '"', '')) AS nombre_hospital,
    LOWER(REPLACE(TRIM(nombre_funcional_completo), '"', '')) AS nombre_funcional_completo,
    LOWER(REPLACE(TRIM(nombre_generico), '"', '')) AS nombre_generico,
    LOWER(REPLACE(TRIM(especialidad), '"', '')) AS especialidad,
    LOWER(REPLACE(TRIM(tipo_atencion), '"', '')) AS tipo_atencion,
    LOWER(REPLACE(TRIM(direccion), '"', '')) AS direccion,
    LOWER(REPLACE(TRIM(barrio), '"', '')) AS barrio,
    LOWER(REPLACE(TRIM(fuente_datos), '"', '')) AS fuente_datos,
    comuna,
    REPLACE(TRIM(telefono), '"', '') AS telefono,
    REPLACE(TRIM(sitio_web), '"', '') AS sitio_web,
    'hospital' AS tipo_entidad,
    geometry,
    CAST(st_x(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS DOUBLE) AS longitud,
    CAST(st_y(st_transform(st_geomfromtext(geometry, 9498), 4326)) AS DOUBLE) AS latitud
  FROM proyecto_final.raw.hospitales_bronze
  WHERE nombre_hospital IS NOT NULL AND geometry IS NOT NULL
),
datos_deduplicados AS (
    SELECT *,
      CONCAT('POINT (', longitud, ' ', latitud, ')') AS coordenadas,
      ROW_NUMBER() OVER (PARTITION BY nombre_hospital, geometry ORDER BY nombre_hospital ) AS rn
    FROM datos_limpios
)
SELECT 
  nombre_hospital,
  nombre_funcional_completo,
  nombre_generico,
  especialidad,
  tipo_atencion,
  direccion,
  CASE WHEN barrio= 'boca' THEN 'la boca' ELSE barrio END AS barrio,
  fuente_datos,
  comuna,
  telefono,
  sitio_web,
  coordenadas,
  tipo_entidad,
  longitud,
  latitud
FROM datos_deduplicados
WHERE rn = 1;